# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せる(2026-11-08 に完了。ito4・auth・share・html は橙・admin・ssh・mail は灰) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |
| [19-security-fixes-2026-10-06](19-security-fixes-2026-10-06.ipynb) | 2026-10-05 の診断の反映(直したもの・確かめたこと・残り) |
| [20-indoor-positioning](20-indoor-positioning.ipynb) | 現在地の測位の再構築(候補 → 相互検証 → 粒子フィルタ・パラメータ・比較実験・RTT・ARCore) |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せる(2026-11-08 に完了。ito4・auth・share・html は橙・admin・ssh・mail は灰) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 17. Cloudflare に載せる(2026-10-06 の計画を 2026-11-08 に書き直し。**2026-11-08 に完了**: 配備・ネームサーバー・4 つの名前を橙・WAF の国のルールまで)

利用者(11-08)「ito4.jp を Cloudflare に移行します」。10-06 の計画(当時は「書くだけ」)を、いまの作りに合わせて書き直した。
**決めるのも、DNS・登録業者・Cloudflare の画面を触るのも利用者。** こちらはサーバーと PC の側を用意した(配備は利用者)。

## 0. 結論(先に)

- **ポートの壁はもう無い。** 外から開いているのは **22・80・443 だけ**(11-08 に本番で `ss -ltn` を読んだ)。
  10-06 に問題だった口は全部片付いている: 3001(Logto)・6001(Soketi)は 10-19 に閉じて Logto は `auth.ito4.jp:443` へ、
  Soketi は 11-02 に撤去、3002(Logto Console)・8281(phpMyAdmin)は 127.0.0.1 だけにして SSH のトンネルで開く。
  80・443 は Cloudflare が中継できるポート
- **ただし、名前を橙(中継)にする前に直さないと止まるものが 3 つあった。** どれも用意した(§2)。**灰のうちに配備しても何も変わらない**
  1. **SSH** —— 配備・控え・%%host・トンネルは全部 `…@ito4.jp` に繋ぐ。Cloudflare は 22 番を中継しない → 灰の名前 `ssh.ito4.jp` へ曲げる
  2. **国の拒否** —— 日本だけ通す反転のままだと、橙の送り元(Cloudflare の IP。多くは日本の割り当てではない)が全部落ち、**公開ページが丸ごと止まる**
     → Cloudflare の範囲を国で落とす前に通す(`ALLOW_CLOUDFLARE=yes`)。国で絞るのは Cloudflare の WAF へ
  3. **本当の送り元** —— 直さないと、回数の制限は全員まとめて 1 人扱い、IP の許可・訪問者の記録も Cloudflare の IP を見る
     → nginx の `set_real_ip_from`(Cloudflare の範囲だけ)+ `real_ip_header CF-Connecting-IP`
- **無料の版は 1 回の本文が 100MB まで。** APK の差し替え(`/admin/downloads.php`、最大 210MB)は**灰の admin.ito4.jp** で行う(今と同じ)。
  HTML 展開の ZIP(最大 20MB)は橙でも通る
- 進め方は「名前ごとに少しずつ橙にする」。**いつでも灰に戻せば元どおり**(§4)

> 中継できるポート・本文の上限・WAF の条件は Cloudflare の公開資料による。**当てる前に Cloudflare の画面と最新の資料で確かめること。**
> Cloudflare の範囲は 2026-11-08 に https://www.cloudflare.com/ips-v4・ips-v6 を読んだ値(IPv4 15・IPv6 7)。

## 1. 名前の分け方

| 名前 | 中継 | 理由 |
|---|---|---|
| `ito4.jp`(公開ページ・アプリの API・共有のリンク・App Links) | **橙**(最初に) | 本体。帯域と攻撃を肩代わりさせ、国の拒否を WAF で行う |
| `auth.ito4.jp`(Logto のサインイン) | 橙(ito4.jp が落ち着いてから) | 443 だけなので中継できる。Logto へは nginx が `X-Forwarded-For $remote_addr`(= 本当の送り元)を渡す |
| HTML 展開の名前・`share.ito4.jp` | 橙(同上) | 443 だけ。ZIP は 20MB まで |
| `admin.ito4.jp`(管理画面) | **灰のまま** | IP で絞っている(allow-admin・タスクマネージャー)。APK の 210MB は中継の上限を超える |
| **`ssh.ito4.jp`(新しく作る)** | **灰(必ず)** | A レコード = 本番の IP。この PC の SSH の行き先(§2-1)。橙にすると 22 番が届かない |
| `mail.ito4.jp`・MX・SPF/DKIM/DMARC の TXT・CAA | **灰(そのまま)** | メールは中継しない。TXT・CAA は中継とは関係ないが、取り込みで抜けていないかを必ず見る |

- **灰の名前が残る限り、本番の IP は見える**(admin・ssh・mail)。橙にしても IP を直に叩かれる道は残る。
  そちらには**今までどおりホストの国の拒否が効く**(Cloudflare の範囲以外の送り元は、日本でなければ落ちる)
- 「80・443 を Cloudflare からだけ受ける」(10-06 の案の手順 6)は**今はできない**: 灰の `admin.ito4.jp` と `auth`(灰のうち)が**同じ 443** を使うので、
  ホストの nftables で絞ると管理画面まで落ちる。やるなら nginx で名前ごとに `$realip_remote_addr`(中継の元の IP)を見る形になる(別の段)

## 2. 用意したもの(2026-11-08。check.php の「国の拒否」「Cloudflare」の行で見張る)

### 2-1. SSH —— `scripts/ssh-host-alias.ps1`(この PC。配備しない)

スクリプトの既定の名前(十数本)・登録したタスクの引数・デスクトップの cmd を全部書き換える代わりに、**この PC の `~\.ssh\config` で 1 回だけ曲げる**:

```
Host ito4.jp
    HostName ssh.ito4.jp      ← 灰の名前。本番の IP を答える
    HostKeyAlias ito4.jp      ← known_hosts の今の ito4.jp の行をそのまま使う
```

ssh も scp も `~\.ssh\config` を読むので、配備・控え(日曜 3:00)・update-containers・%%host・「トンネル開通SSH.cmd」(`km@ito4.jp`)が**何も変えずに**灰の名前へ行く。
ホスト鍵の照合は今までどおり ito4.jp の行で行う(相手が違えば止まる)。

| 叩き方 | すること |
|---|---|
| `.\ssh-host-alias.ps1` | 見るだけ(DNS の答えが橙か灰か・段の有無・ssh の行き先・known_hosts) |
| `.\ssh-host-alias.ps1 -Apply` | 段を `~\.ssh\config` の先頭に足す。**`ssh.ito4.jp` が灰で引けなければ書かない**。前のファイルは `~\.ssh\Old\config-<日時>` |
| `.\ssh-host-alias.ps1 -Test` | kmops の鍵で 1 回繋いで `echo` を返させる(読むだけ) |
| `.\ssh-host-alias.ps1 -Undo` | 段を外す(同じく写してから) |

2026-11-08 に写しの config で試した: 段の付け外しで元のファイルと同じに戻る・2 回 -Apply しても段は 1 つ・
`HostName admin.ito4.jp`(灰)に曲げて `kmops@ito4.jp` へ繋がる(HostKeyAlias で今の known_hosts のまま通る)。

### 2-2. 国の拒否 —— `scripts/host-geoblock.sh` の `ALLOW_CLOUDFLARE`(ホスト。設定は `geoblock.local.conf`)

- `ALLOW_CLOUDFLARE=yes` で、Cloudflare の範囲(`https://www.cloudflare.com/ips-v4`・`ips-v6`)を **`CLOUDFLARE_PORTS`(既定 80 443)だけ**、国で落とす前に通す
- 一覧は `update`(毎月 2 日の cron も)で取り、`/var/lib/kosenmap/geoblock/cf.v4`・`cf.v6` に置く。**取れない・形が違えば前の一覧を使う**
- **yes なのに一覧が無ければ `apply` は当てない**(橙の名前に来る全員を落とすため)
- 取った一覧を `nginx/km/cloudflare-real-ip.conf` の `set_real_ip_from` と見比べ、違えば注意を出す(`update`・`status`)
- BAN の集計(`host-stats.sh`)は、Cloudflare を通した数を「落とした数」にも「Google」にも数えない
- **ALLOW_GOOGLE は残す**(灰の名前への Google の取得に要る。橙の ito4.jp の App Links は Cloudflare 経由で届く)

### 2-3. 本当の送り元 —— `nginx/km/cloudflare-real-ip.conf`(配備で載る)

- `default.conf.template` の http の段(回数の制限と server より前)で include する
- `set_real_ip_from` は **Cloudflare の範囲だけ**(IPv4 15・IPv6 7)。そこから来た接続のときだけ `CF-Connecting-IP` を読むので、
  **直に来た相手が CF-Connecting-IP を付けても無視される**。`real_ip_recursive` は off のまま
- `$remote_addr` が本当の送り元になるので、回数の制限・`allow-*.conf`・`X-Real-IP`(PHP の回数制限・管理の記録)・訪問者の記録・ログが**そのまま正しく働く**(PHP は直さなくてよい)
- 公式の nginx の像に realip が入っていることを 11-08 に本番で確かめた(`nginx -V`)。**この PC には docker が無いので `nginx -t` はまだ通していない** → 配備のあと §3 の 0 で確かめる

### 2-4. Cloudflare の画面で決めること(利用者)

| 項目 | 勧め | 理由 |
|---|---|---|
| SSL/TLS | **Full (strict)** | サーバーは Let's Encrypt の正規の証明書を持っている。Flexible は中継とサーバーの間が平文になる |
| Always Use HTTPS | **切** | 入れると 80 番の `/.well-known/acme-challenge/` まで 443 へ曲げられ、443 の server は確認のファイルを返さないので**証明書の更新が失敗する**。80→443 はサーバーが自分で行っている |
| Bot Fight Mode・Browser Integrity Check | **切**(最初は) | アプリの API・Google の App Links の確認(assetlinks.json)・証明書の確認を弾くことがある |
| 国の拒否(WAF のカスタムルール) | 日本以外を Block。**ただし** `/.well-known/assetlinks.json` と `/.well-known/acme-challenge/` は除く | ホストの反転(日本だけ)と同じ振る舞い。除く 2 つは海外(Google・Let's Encrypt)から来る |
| キャッシュ | 既定のまま | HTML・PHP・JSON の口は既定でキャッシュされない。静的ファイルは `?v=` 付きなので古いものは出ない |
| CAA | 確かめる | 今は `0 issue "letsencrypt.org"` だけ。橙の名前に Cloudflare が出す証明書の CA を許す必要がある(Cloudflare が自動で足すかを画面で見る) |

## 3. 手順(上から順に。**5 までは中身が何も変わらない**)

0. **配備する**(灰のうちは何も変わらない)。そのあと下の %%host のセルで、nginx が新しい設定を読んでいることを確かめる
   ```
   pwsh -File C:\Users\itota\Documents\Website\server\scripts\deploy-to-host.ps1 -Backup -Yes
   ```
   続けて nginx を起こし直す(**これを忘れると realip が効かない**。テンプレートは 1 ファイルの mount なので、配備で置き換えても動いている nginx には古い中身が見えたまま。11-08 に実際に起きた):
   ```
   pwsh -File C:\Users\itota\Documents\Website\server\scripts\deploy-to-host.ps1 -Action restart -Services reverse-proxy
   ```
1. **国の拒否に Cloudflare を足す**(km で sudo)。`geoblock.local.conf` に `ALLOW_CLOUDFLARE=yes` と `CLOUDFLARE_PORTS=80 443` を足して:
   ```
   cd /opt/kosenmap
   sudo sh scripts/host-geoblock.sh --path /opt/kosenmap update
   sudo sh scripts/host-geoblock.sh --path /opt/kosenmap apply --dry-run
   sudo sh scripts/host-geoblock.sh --path /opt/kosenmap apply
   sudo sh scripts/host-geoblock.sh --path /opt/kosenmap status
   ```
   `update` が「cf: nginx の set_real_ip_from と同じです」と出すこと。`apply --dry-run` の表に `@cf4`・`@cf6` の accept が**国の drop より前**にあること
2. **今の DNS の業者で `ssh.ito4.jp` の A レコード(本番の IP)を作る。** ネームサーバーを替える前から使える
3. **この PC:** `pwsh -File C:\Users\itota\Documents\Website\server\scripts\ssh-host-alias.ps1 -Apply` → `-Test`(OK が出ること)。このあと配備や控えが `ssh.ito4.jp` を通る
4. **Cloudflare に `ito4.jp` を足し、今の DNS を取り込む。全部「灰」のまま。** 取り込みで抜けやすいものを並べて見比べる:
   MX・SPF/DKIM/DMARC の TXT・CAA・`admin`・`auth`・`share`・HTML 展開の名前・`mail`・`ssh`
   (11-08 の取り込みでは `html.ito4.jp` が抜け、証明書の更新の下見が NXDOMAIN で落ちた。MX は元から無い)
5. **登録業者でネームサーバーを Cloudflare に替える。** DNSSEC を使っているなら、先に登録業者で切る(替えたあと Cloudflare で入れ直す)。
   下の %%ps のセルで、Cloudflare と Google の公開 DNS が同じ答えを返すことを確かめる
6. Cloudflare の画面で §2-4 を決める(Full (strict)・Always Use HTTPS 切・Bot Fight Mode 切・WAF の国のルール)
7. **`ito4.jp` を橙にする。** 確かめること:
   - 公開ページ・アプリの地図の取得・共有のリンクがアプリで開く(端末の `pm get-app-links` が verified)
   - 管理画面の訪問者の記録に**本当の IP** が出る(Cloudflare の 104.x・172.6x・162.15x などが並ばない)
   - 回数の制限が 1 人ずつ効く(1 台からの連打で 429、ほかの人は通る)
   - `pwsh -File C:\Users\itota\Documents\Website\server\scripts\ssh-host-alias.ps1` が「ito4.jp 橙・ssh.ito4.jp 灰」・配備の下見が通る
   - 証明書の更新: ホストで `sudo sh scripts/host-cert.sh --path /opt/kosenmap renew --dry-run`
8. 落ち着いたら `auth`・`share`・HTML 展開の名前を **1 つずつ**橙にする(`admin`・`ssh`・`mail` は灰のまま)

## 4. 戻し方

- **名前ごとに灰へ戻すだけ**で、DNS の答えは本番の IP に戻る(数分で効く)
- `ALLOW_CLOUDFLARE=yes`・nginx の realip・SSH の段は**戻さなくてよい**(灰に戻ると Cloudflare からの接続が来なくなるだけ。`ssh.ito4.jp` が本番を指す限り SSH も通る)。
  外すなら `ALLOW_CLOUDFLARE=no` → `apply`、`ssh-host-alias.ps1 -Undo`
- ネームサーバーを元の業者へ戻すのは最後の手段(反映に最大 48 時間ほどかかりうる。その間も灰の記録が同じなら害は無い)

## 5. まだ確かめていないこと(当てる前・当てながら)

- **`nginx -t`**: この PC に docker が無いので、新しい include はまだ nginx に読ませていない → §3 の 0 のセルで確かめる(検証機 192.168.3.31 が立てば先にそこでも)
- 学校のネットワーク(プロキシ・フィルタ)から Cloudflare の名前に届くか
- CAA と、Cloudflare が橙の名前に出す証明書の CA
- 無料の版の WAF のカスタムルールの数と、国の条件が使えるか
- Cloudflare の範囲が変わったとき → `host-geoblock.sh update`(毎月)が見比べて注意を出す。直すのは `nginx/km/cloudflare-real-ip.conf` と配備

In [ ]:
%%host
# §3 の 0: 配備のあと、nginx が Cloudflare の本当の送り元の設定を読んでいるか(読むだけ)
bad=0
if [ -f nginx/km/cloudflare-real-ip.conf ] && grep -q 'include /etc/nginx/km/cloudflare-real-ip.conf;' nginx/default.conf.template; then
  echo "OK   配備済み(set_real_ip_from $(grep -c '^set_real_ip_from' nginx/km/cloudflare-real-ip.conf) 本)"
else
  echo "★    まだ配備していません(nginx/km/cloudflare-real-ip.conf が無い)"; bad=1
fi
if docker compose exec -T reverse-proxy nginx -t 2>&1 | grep -q 'test is successful'; then
  echo "OK   nginx -t が通る"
else
  echo "★    nginx -t が通りません:"; docker compose exec -T reverse-proxy nginx -t 2>&1 | tail -n 5; bad=1
fi
if docker compose exec -T reverse-proxy nginx -T 2>/dev/null | grep -q '^real_ip_header CF-Connecting-IP;'; then
  echo "OK   動いている nginx が real_ip_header CF-Connecting-IP を読んでいる"
else
  echo "★    動いている nginx はまだ読んでいません(reverse-proxy を入れ直す: 配備の -Action up)"; bad=1
fi
exit $bad

In [ ]:
%%ps
# §3 の 5・7: 公開の DNS 2 か所の答えと、この PC の SSH の行き先(読むだけ)
$names = 'ito4.jp', 'auth.ito4.jp', 'admin.ito4.jp', 'share.ito4.jp', 'mail.ito4.jp', 'ssh.ito4.jp'
foreach ($s in '1.1.1.1', '8.8.8.8') {
    foreach ($n in $names) {
        try {
            $ip = (Resolve-DnsName $n -Type A -Server $s -DnsOnly -ErrorAction Stop | Where-Object Type -eq 'A').IPAddress -join ','
            '{0,-8} {1,-14} → {2}' -f $s, $n, $ip
        } catch { '{0,-8} {1,-14} → 引けない({2})' -f $s, $n, $_.Exception.Message }
    }
    try {
        $mx = (Resolve-DnsName ito4.jp -Type MX -Server $s -DnsOnly -ErrorAction Stop | Where-Object Type -eq 'MX').NameExchange -join ','
        '{0,-8} {1,-14} → {2}' -f $s, 'MX', $(if ($mx) { $mx } else { '(なし)' })
    } catch { '{0,-8} MX 引けない' -f $s }
}
# 橙か灰か・SSH の段・known_hosts(scripts/ssh-host-alias.ps1 は何も変えない)
pwsh -NoProfile -File .\ssh-host-alias.ps1

## 6. Cloudflare の分析を取り込む(2026-11-08。訪問者の画面に出す)

利用者「GraphQL Analytics API は Free プラン OK。Logpush は Free ではだめ。作ってください(管理画面の訪問者に紐づける)」。

**取り込むもの:** WAF で止めた記録(サーバーまで届かない分。IP・国・ASN・パス・ルール)と、時間ごとの通信量(要求・キャッシュ・転送量・脅威・国)。
**消さない**(利用者「いろいろ調査したいので、位置の保存は永久に」)。Cloudflare 側は細かいデータを 31 日ほどしか持たないので、毎時取って貯める。初回は持っている範囲(最大 30 日)をさかのぼる。

| 部品 | 中身 |
|---|---|
| `scripts/host-cloudflare-import.sh` | root の cron(**版 15・毎時 13 分**。版 14 は `--label` の空白で毎回止まっていた。11-13 に直した)。curl で `api.cloudflare.com/client/v4/graphql` へ。**トークンは web に渡さない**(root だけが読む `cloudflare.local.conf`・`-H @ファイル`)。同じ失敗は 6 時間に 1 回だけメール |
| `src/scripts/cloudflare-import.php` | web の中。`plan`(DB の続きから問い合わせを作る)・`ingest`(答えを読み解いて DB へ) |
| `src/lib/cloudflare-analytics.php` | 表 `km_cf_waf_events`・`km_cf_traffic_hourly`・`km_cf_import_state`(新しい表だけ) |
| `src/lib/visitor-geo.php` | 訪問者の位置を DB に(`km_visitor_geo`・`km_visitor_geo_history`)。WAF の記録の国・ASN は、まだ無い IP にだけ入れる |
| 画面 | 訪問者 →「Cloudflare 全体(日別)」「止めた国」「止めた接続」(同じ IP の訪問と判定を並べる・CSV)・上のタイル「Cloudflare で止めた」 |

### 手順(利用者)

1. 配備する(web の新しい口・表・台本)。`-Action up` は要らない
   ```
   pwsh -File C:\Users\itota\Documents\Website\server\scripts\deploy-to-host.ps1 -Backup -Yes
   ```
2. **Cloudflare の画面で API トークンを作る:** 右上の人の印 → プロフィール → API トークン → トークンを作成 → カスタム トークン
   - 権限: **ゾーン → Analytics → 読み取り**(これだけ)
   - ゾーン リソース: 特定のゾーン → `ito4.jp`
   - クライアント IP アドレスのフィルタ: 本番の IP だけ(任意。漏れてもほかの所からは使えない)
3. **ゾーンの ID** を控える(ito4.jp の「概要」の右下・32 桁の 16 進)
4. km で設定を置く(**root の持ち物・600**。値は画面に出さない・チャットに貼らない)
   ```
   cd /opt/kosenmap
   sudo install -m 600 -o root -g root cloudflare.local.conf.example cloudflare.local.conf
   sudo nano cloudflare.local.conf
   ```
5. km で cron を版 15 にして、1 回手で流す(**11-13**: 版 14 の行は `--label Cloudflare の取り込み` の空白で `send-log.sh` が「知らない引数」で止まり、11-08 の手の 1 回のあと取り込めていなかった。版 15 で `Cloudflare取り込み` に)
   ```
   sudo sh scripts/host-updates-setup.sh --path /opt/kosenmap --fix
   sudo sh scripts/host-cloudflare-import.sh --path /opt/kosenmap
   ```
   「済みました(問い合わせ N 回)」が出ること。初回は 30 日ぶんをさかのぼるので数十回になる
6. 管理画面の訪問者を開き、Cloudflare のカードに「最後に取り込んだ」と、止めた接続が出ること

### 確かめていないこと(11-08)

- **実の DB と実の API ではまだ動かしていない**(手元に MariaDB とトークンが無い)。項目の名前は公式の例と、利用者が画面から書き出した WAF の記録の形に合わせた。
  もし Cloudflare が項目の名前を断ったら、画面の「★ 取り込みに失敗」に Cloudflare の文がそのまま出る → それを貼ってもらって直す
- 無料の版で `httpRequests1hGroups` がどこまでさかのぼれるか(settings の notOlderThan を読んで合わせる。読めなければ控えめに 7 日)

## 7. 案内AIのトンネル(Cloudflare Tunnel。2026-11-12・段 12)

利用者(11-11)「トンネルを経由する方針で行きましょう」。案内チャットの LLM(自宅 PC)を、ngrok の代わりに Cloudflare Tunnel で出す。

```
VPS の PHP ─ https ─ Cloudflare(llm.ito4.jp。WAF で本番サーバーの IP だけ)
           ─ トンネル ─ cloudflared(自宅 PC)─ 門番 server.py(127.0.0.1)─ Unsloth / Bionic
```

- **替えた理由**: ngrok のエンドポイントの期限切れで止まった(11-07)・ito4.jp はすでに Cloudflare を通っている(質問と答えを見られる会社が 1 つ減る)・WAF で送り元を絞れる
- **自分で管理するトンネル**(locally-managed)。Zero Trust の画面は使わない。トンネルの鍵は自宅 PC の `%USERPROFILE%\.cloudflared\<UUID>.json`(**秘密**。git に入らない)
- **使うときだけ起こす**(`guide-llm/start-guide-llm.ps1`。Windows のサービスにはしない)
- 守りは 3 つ: WAF(本番サーバーの IP だけ)・トンネルの行き先(1 つの名前だけ。ほかは 404)・門番の鍵(`KM_GUIDE_LLM_KEY`。無ければ 401)
- 待ち時間: PHP は全部で 55 秒・nginx は 90 秒。Cloudflare の中継の上限(100 秒)に収まる
- **2026-11-12 に通しで確かめた**: cloudflared は QUIC でつながる・本番から門番に届く・管理画面「届くか確かめる」が 0.1 秒で OK。古いつなぎ(ngrok)の分岐は同じ日に消した

### 7-1. 作り方(利用者。1 回だけ)

自宅 PC で:

```
winget install --id Cloudflare.cloudflared
cloudflared tunnel login                                  ← ブラウザが開く。ito4.jp を選ぶ
cloudflared tunnel create kosenmap-guide                  ← UUID が出る
cloudflared tunnel route dns kosenmap-guide llm.ito4.jp   ← 橙の CNAME が 1 つできる
```

`guide-llm/guide-llm.env` に足す(見本は `guide-llm.env.example`):

```
CLOUDFLARE_TUNNEL=<UUID>
CLOUDFLARE_HOSTNAME=llm.ito4.jp
```

Cloudflare の画面 → セキュリティ → WAF → カスタムルール:

| ルール | 式 | 動き |
|---|---|---|
| **足す(いちばん上)** | `(http.host eq "llm.ito4.jp" and ip.src ne <本番サーバーの IP>)` | Block |
| 国のルール(§2-4)に足す | 今の式に `and http.host ne "llm.ito4.jp"` | (Block のまま) |

- **本番サーバーの IP はここに書かない**(この文書は公開の控えに入る)。`ssh.ito4.jp` が答える IP
- 国のルールに足すのは、サーバーからの呼び出しを国で落とさないため(上のルールが先に効くので、ほかの国からは届かない)

本番の `.env` の `KM_GUIDE_LLM_URL` を `https://llm.ito4.jp` にして配備(env を変えるので `-Action up`):

```
pwsh -File C:\Users\itota\Documents\Website\server\scripts\deploy-to-host.ps1 -Backup -Yes -Action up
```

### 7-2. 確かめ方

- 自宅 PC で `pwsh -File guide-llm\start-guide-llm.ps1` → cloudflared が `Registered tunnel connection` を出す
- WAF のルールを入れる前: 外から `https://llm.ito4.jp/health` は 200・鍵なしの `/v1/models` は 401
- WAF のルールを入れた後: 本番サーバー以外からは 403(Cloudflare が止める)。下のセルで本番から 200 が返ること
- 管理画面「案内チャット」の「届くか確かめる」

### 7-3. つながらないとき・戻し方

- cloudflared は UDP(QUIC)でつなぎ、だめなら http2 に落ちる。セキュリティのソフトが間に入るときは `guide-llm.env` に `CLOUDFLARE_PROTOCOL=http2`
  (ngrok では Norton の TLS 検査が引っかかった。11-07)
- 止める: 本番の `.env` の `KM_GUIDE_LLM_URL` を空にして `-Action up`(案内は言葉で探した結果だけになる)。ngrok へは戻せない(分岐を消した。要るなら Old ではなく GitHub の控えの 11-12 より前の版)
- やめる: `cloudflared tunnel delete kosenmap-guide`・DNS の `llm` の CNAME を消す・WAF のルールを外す

In [ ]:
%%host
# §7-2: 本番の web のコンテナから、案内AIの門番に届くか(読むだけ。鍵は送らない・値も出さない)
url=$(docker compose exec -T web printenv KM_GUIDE_LLM_URL 2>/dev/null | tr -d '\r')
if [ -z "$url" ]; then
  echo "－    KM_GUIDE_LLM_URL が空です(案内は言葉で探した結果だけ出します)"; exit 0
fi
case "$url" in
  *ngrok*) echo "注意 まだ古いつなぎ(ngrok)の URL です" ;;
  *)       echo "OK   向き先: $url" ;;
esac
code=$(docker compose exec -T web curl -s -o /dev/null -m 10 -A 'KosenMap-Guide/1' -w '%{http_code}' "$url/health" 2>/dev/null)
case "$code" in
  200) echo "OK   門番が答えました(200)" ;;
  403) echo "★    Cloudflare が止めました(403)。WAF のルールの IP が本番サーバーのものかを確かめる" ;;
  530|502|000) echo "－    届きません($code)。自宅 PC の start-guide-llm.ps1 が動いていないとき(ふだんは止めてある)" ;;
  *)   echo "★    思わぬ答え($code)" ;;
esac